# Data Cleaning

Hey Ivette and Julia! As we discussed, I specifically loaded the bank-full.csv file and took the following steps for cleaning:


*   Checked for NA values (there were none to remove)
*   Converted all categorical variables to numeric
*   Defined X_bank_full and y_bank_full as features and target subsets of the bank_full dataframe

I figured for each algorithm I would let each of us do our own training and test set split, unless there's a good reason for us to use the same split. Let me know if I can answer any questions!

-Emmet

In [1]:
# Import statements
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

import numpy as np; np.set_printoptions(precision=2)
import matplotlib.pyplot as plt; #plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams.update({ "figure.figsize": [8, 3], "figure.dpi": 125, "text.usetex": True, "font.family": "Helvetica" })
import pandas as pd; pd.options.display.float_format = "{:,.2f}".format
import warnings; warnings.filterwarnings('ignore')

In [2]:
!pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# metadata
print(bank_marketing.metadata)

# variable information
print(bank_marketing.variables)

{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [4]:
import pandas as pd

In [6]:
bank_full = pd.read_csv('bank-full.csv', sep = ';')
print(bank_full.head())

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  


In [7]:
# Count NA/NULL values in each column (code from: https://www.geeksforgeeks.org/how-to-count-the-number-of-nan-values-in-pandas/)
column_nan_count = bank_full.isna().sum()
print("NaN count per column:")
print(column_nan_count)

# Appears to be no NA/NULL values in bank_full

NaN count per column:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [8]:
# Identify unique labels in categorical columns
for column in list(bank_full.columns):
  unique_values= bank_full[str(column)].unique()
  print('Unique values in ' + str(column) + ':' + str(unique_values))

Unique values in age:[58 44 33 47 35 28 42 43 41 29 53 57 51 45 60 56 32 25 40 39 52 46 36 49
 59 37 50 54 55 48 24 38 31 30 27 34 23 26 61 22 21 20 66 62 83 75 67 70
 65 68 64 69 72 71 19 76 85 63 90 82 73 74 78 80 94 79 77 86 95 81 18 89
 84 87 92 93 88]
Unique values in job:['management' 'technician' 'entrepreneur' 'blue-collar' 'unknown'
 'retired' 'admin.' 'services' 'self-employed' 'unemployed' 'housemaid'
 'student']
Unique values in marital:['married' 'single' 'divorced']
Unique values in education:['tertiary' 'secondary' 'unknown' 'primary']
Unique values in default:['no' 'yes']
Unique values in balance:[ 2143    29     2 ...  8205 14204 16353]
Unique values in housing:['yes' 'no']
Unique values in loan:['no' 'yes']
Unique values in contact:['unknown' 'cellular' 'telephone']
Unique values in day:[ 5  6  7  8  9 12 13 14 15 16 19 20 21 23 26 27 28 29 30  2  3  4 11 17
 18 24 25  1 10 22 31]
Unique values in month:['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'jan' 'feb' 'mar' 'apr

In [9]:
# Convert categorical variables with integers
bank_full['job'].replace(['admin.', 'blue-collar', 'entrepreneur', 'housemaid',
                          'management', 'retired', 'self-employed', 'services',
                          'student', 'technician', 'unemployed', 'unknown'],
                           [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], inplace=True)
bank_full['marital'].replace(['single', 'married', 'divorced'],[1, 2, 3], inplace=True)
bank_full['education'].replace(['tertiary', 'secondary', 'unknown', 'primary'],[3, 2, 0, 1], inplace=True)
bank_full['default'].replace(['no', 'yes'], [0, 1], inplace=True)
bank_full['housing'].replace(['yes', 'no'], [1, 0], inplace=True)
bank_full['loan'].replace(['no', 'yes'], [0, 1], inplace=True)
bank_full['contact'].replace(['unknown', 'cellular', 'telephone'], [0, 1, 2], inplace=True)
bank_full['month'].replace(['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], inplace=True)
bank_full['poutcome'].replace(['unknown', 'failure', 'other', 'success'], [2, 0, 3, 1], inplace=True)
bank_full['y'].replace(['no', 'yes'], [0, 1], inplace=True)

In [10]:
# Verifying that variable replacement worked
print(bank_full.head())

   age  job  marital  education  default  balance  housing  loan  contact  \
0   58    5        2          3        0     2143        1     0        0   
1   44   10        1          2        0       29        1     0        0   
2   33    3        2          2        0        2        1     1        0   
3   47    2        2          0        0     1506        1     0        0   
4   33   12        1          0        0        1        0     0        0   

   day  month  duration  campaign  pdays  previous  poutcome  y  
0    5      5       261         1     -1         0         2  0  
1    5      5       151         1     -1         0         2  0  
2    5      5        76         1     -1         0         2  0  
3    5      5        92         1     -1         0         2  0  
4    5      5       198         1     -1         0         2  0  


In [11]:
# define features and targets dataframes
X_bank_full = bank_full[['age','job','marital','education','default','balance','housing','loan','contact','day','month','duration','campaign','pdays','previous','poutcome']]
y_bank_full = bank_full[['y']]

print(X_bank_full.head())
print(y_bank_full.head())

   age  job  marital  education  default  balance  housing  loan  contact  \
0   58    5        2          3        0     2143        1     0        0   
1   44   10        1          2        0       29        1     0        0   
2   33    3        2          2        0        2        1     1        0   
3   47    2        2          0        0     1506        1     0        0   
4   33   12        1          0        0        1        0     0        0   

   day  month  duration  campaign  pdays  previous  poutcome  
0    5      5       261         1     -1         0         2  
1    5      5       151         1     -1         0         2  
2    5      5        76         1     -1         0         2  
3    5      5        92         1     -1         0         2  
4    5      5       198         1     -1         0         2  
   y
0  0
1  0
2  0
3  0
4  0


# Logistic Regression Code

In [18]:
# Needed Libraries:

# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
from sklearn.model_selection import train_test_split

# There was a sepereate Geeks4Geeks article on this to standardize categorical data too
# I can't find the link however
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# https://scikit-learn.org/stable/modules/compose.html
# https://www.geeksforgeeks.org/pipelines-python-and-scikit-learn/
from sklearn.pipeline import Pipeline

# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
from sklearn.linear_model import LogisticRegression

# https://scikit-learn.org/stable/api/sklearn.metrics.html
from sklearn.metrics import accuracy_score, classification_report, precision_score

# Included Pandas
import pandas as pd


In [19]:
# Copying this over from the API hints in the midterm
def train_and_evaluate_model(model, X_train, X_test, y_train, y_test):
    # Train model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate precision
    precision = precision_score(y_test, y_pred)

    # Print detailed classification report
    print(f"\nModel: {model.__class__.__name__}")
    print(f"Precision Score: {precision:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return precision


In [22]:
# Copying the entire structure over from the API hints from the midterm - prepping the data

# Convert categorical features to numerical (since I again was copying code from the midterm)
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] # List of categorical columns
numerical_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] # List of numerical columns

# Create a ColumnTransformer to apply different preprocessing to different columns
# Stolen from stackoverflow https://stackoverflow.com/questions/54160370/how-to-use-sklearn-column-transformer
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# https://www.geeksforgeeks.org/ml-one-hot-encoding/
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features),
    ])

# Feature selection, kind of threw everything in there due to lack of domain knowledge
# And also because we're going to use L2 regularization anyways
features = ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']
y_bank_full = bank_full[['y']]

In [23]:
# Splitting the data into a training/test set

X_train, X_test, y_train, y_test = train_test_split(X_bank_full, y_bank_full, test_size=0.2, random_state=42) # Use X_bank_full instead of X

# Fit and transform the training data
X_train_scaled = preprocessor.fit_transform(X_train)

# Transform the test data
X_test_scaled = preprocessor.transform(X_test)

In [31]:
# Training the model, this is copied over from the midterm as well
model = LogisticRegression(random_state=42, penalty='l2', C=1.0)  # C = strength of regularization

# Train and evaluate the models
results = train_and_evaluate_model(model, X_train_scaled, X_test_scaled, y_train.values.ravel(), y_test.values.ravel())

# Printing out the results - removed the max function call here
# and just print out the precision.
print(f"\nLogistic Regression with precision {results:.3f}")



              precision    recall  f1-score   support

           0       0.92      0.98      0.94      7952
           1       0.65      0.34      0.45      1091

    accuracy                           0.90      9043
   macro avg       0.78      0.66      0.70      9043
weighted avg       0.88      0.90      0.88      9043


Logistic Regression with precision 0.899


In [29]:
# This is extra to check out the different C values/what's the best one, if L2 is
# better (it absolutely makes more sense but still)

# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html
# Recommended by GeeksForGeeks
# https://www.geeksforgeeks.org/hyperparameter-tuning-using-gridsearchcv-and-kerasclassifier/

from sklearn.model_selection import GridSearchCV


# Define the parameter grid
parameter_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
}

# Sets up the grid search object
grid_search = GridSearchCV(LogisticRegression(random_state=42), parameter_grid, cv=5, scoring='accuracy')

# Fit the model
grid_search.fit(X_train_scaled, y_train)

# Print out what parameters make the most sense
print("Best parameters found: ", grid_search.best_params_)


Best parameters found:  {'C': 10, 'penalty': 'l2'}


In [33]:
# Get the coefficients
coefficients = model.coef_[0]

# Get the feature names
# had stackoverflow and duckduck go's ai feature help me out with this one
# because I kept getting errors

# Just to be clear
ohe = preprocessor.named_transformers_['cat']
feature_names = numerical_features + list(ohe.get_feature_names_out(categorical_features))

# Create a DataFrame to display features and their coefficients
feature_importance = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})

# Sort by absolute value of coefficients
feature_importance['Absolute Coefficient'] = feature_importance['Coefficient'].abs()
feature_importance = feature_importance.sort_values(by='Absolute Coefficient', ascending=False)

# Print the feature importance
print(feature_importance)


        Feature  Coefficient  Absolute Coefficient
48   poutcome_1         1.52                  1.52
37      month_3         1.51                  1.51
35      month_1        -1.30                  1.30
32    contact_0        -1.23                  1.23
3      duration         1.08                  1.08
45     month_11        -0.91                  0.91
41      month_7        -0.88                  0.88
43      month_9         0.84                  0.84
49   poutcome_2        -0.84                  0.84
44     month_10         0.81                  0.81
47   poutcome_0        -0.72                  0.72
42      month_8        -0.71                  0.71
29    housing_1        -0.61                  0.61
50   poutcome_3        -0.49                  0.49
39      month_5        -0.49                  0.49
46     month_12         0.48                  0.48
31       loan_1        -0.47                  0.47
15        job_9         0.44                  0.44
33    contact_1         0.41   